In [ ]:
"""
=============================================================================================================
Проект: Автоматизация обработки данных ионной хроматографии (Ion Balance Automation)
Данный скрипт - Шаг 1, направленный на извлечение, конвертацию и очистку данных (Data Extraction & Cleaning)
=============================================================================================================

Описание задачи:
Исходные данные, выгружаемые из хроматографа, представляют собой сложную матрицу с двумя строками заголовков
(номер пробы и коэффициент разбавления), а также разбиты по листам (в зависимости от усиления сигнала Gain). 
Данный скрипт преобразует (транспонирует) эту сложную структуру в "плоскую" таблицу (Long format), удобную 
для дальнейшего математического анализа и поиска ионного баланса. Выводимый формат даных вашего прибора может
отличаться от предлагаемого. Для работы скрипта преобразуйте его вручную по аналогии с данными raw_data.xlsx

Входные данные:
- Файл Excel (raw_data.xlsx) с листами усилений кондуктометричесого датчика ('0.01' и '0.1').

Выходные данные:
- Файл Excel (clean_data.xlsx) с колонками: Sample, Ion, Gain, Dilution, Conc (в мг/л).

Особенности реализации:
- Обработка прочерков ('-') и пустых ячеек (замена на NaN и удаление).
- Автоматическое определение коэффициента усиления из названия листа.
- Сохранение исходного порядка колонок (чтобы пробы не сортировались по алфавиту).
- Замена запятых на точки в десятичных дробях для корректной работы Pandas.
================================================================================
"""
import pandas as pd
import numpy as np

# Настройка отображения для Jupyter: показывать все колонки таблицы, если их много
pd.set_option('display.max_columns', None)

def convert_excel_to_clean_excel(file_name, output_name):
    """
    Функция для первичной обработки (ETL) данных хроматографа.
    Превращает сложную матричную таблицу Excel (с двумя шапками и разными листами) 
    в "плоскую" базу данных, удобную для дальнейших математических расчетов.
    """
    print(f"Загрузка файла {file_name}...")
    
    # Читаем сразу все листы из Excel-файла.
    # sheet_name=None возвращает словарь, где ключи - названия листов, а значения - данные.
    # header=None отключает автоматическое чтение первой строки как заголовка, 
    # так как у нас в raw_data.xlsx "двухэтажный" заголовок (в первой строке указан номер пробы,а во второй строке - разбавление).
    excel_data = pd.read_excel(file_name, sheet_name=None, header=None)
    
    clean_rows = []
    
    # Проходимся по каждому листу (например, лист '0.01' и лист '0.1')
    for sheet_name, df in excel_data.items():
        try:
            # Название листа содержит коэффициент усиления (например, '0.01'). 
            # Заменяем запятую на точку на случай, если Excel сохранил название как текст.
            gain = float(sheet_name.replace(',', '.'))
        except ValueError:
            print(f"Внимание: Лист '{sheet_name}' пропущен (название не является числом).")
            continue
        
        # Проходим по всем колонкам (начиная с 1-й, так как 0-я колонка — это названия ионов: F, Cl и т.д.)
        for col_idx in range(1, df.shape[1]):
            sample_id = df.iloc[0, col_idx] # Берем номер пробы из самой первой строки
            dilution = df.iloc[1, col_idx]  # Берем коэффициент разбавления из второй строки
            
            # Если ячейка с номером пробы пустая, пропускаем всю колонку
            if pd.isna(sample_id):
                continue
                
            # Проходим по всем строкам начиная с 3-й (индекс 2), так как первые две — это шапка
            for row_idx in range(2, df.shape[0]):
                ion = df.iloc[row_idx, 0]        # Название иона из первого столбца
                conc = df.iloc[row_idx, col_idx] # Концентрация из текущей ячейки
                
                # Если название иона пустое, пропускаем
                if pd.isna(ion):
                    continue
                    
                # Добавляем данные в виде словаря в общий список
                clean_rows.append({
                    'Sample': str(sample_id).strip(),
                    'Ion': str(ion).strip(),
                    'Gain': gain,
                    'Dilution': dilution,
                    'Conc': conc,
                    'Original_Col_Order': col_idx # сохраняем исходный порядок колонок
                })
                
   # Превращаем список словарей в DataFrame (таблицу)
    result_df = pd.DataFrame(clean_rows)
    
    # --- ОЧИСТКА ДАННЫХ (Data Cleaning) ---
    
    # 1. Замена десятичного разделителя.
    # При работе с русскоязычным Excel числа часто содержат запятую вместо точки 
    # (например, "0,146"). Библиотека Pandas ожидает точку, поэтому принудительно 
    # конвертируем в строку и меняем запятые на точки.
    result_df['Conc'] = result_df['Conc'].astype(str).str.replace(',', '.', regex=False)
    
    # 2. Унификация пустых значений.
    # Приводим все возможные варианты обозначения отсутствия данных (прочерки, пустые строки, 
    # текстовые 'nan') к единому стандартному маркеру NaN (Not a Number), который понимает Pandas.
    result_df['Conc'] = result_df['Conc'].replace(['-', '—', ' ', 'nan', 'None'], np.nan)
    
    # 3. Конвертация в числовой тип.
    # Превращаем текстовые данные в числа. Параметр errors='coerce' страхует нас: 
    # если в колонке окажется нечитаемый текст, он станет NaN, а не вызовет ошибку (Crash).
    result_df['Conc'] = pd.to_numeric(result_df['Conc'], errors='coerce')
    
    # 4. Удаление строк с прочерками (NaN).
    # ВАЖНО: Мы удаляем строки, где концентрация осталась NaN (то есть там были прочерки). 
    # Если этого не сделать, во втором скрипте пустые значения могут быть восприняты как "0", 
    # что приведет к искусственной "подгонке" ионного баланса (скрипт обнулит катионы, 
    # чтобы баланс сошелся). Если ион не измеряли, его просто не должно быть в таблице.
    result_df = result_df.dropna(subset=['Conc'])
    
    # --- СОРТИРОВКА ---
   
    # Сортируем по 'Original_Col_Order', чтобы сохранить физический порядок колонок из Excel 
    result_df = result_df.sort_values(by=['Original_Col_Order', 'Ion'])
    
    # Удаляем техническую колонку с порядком, она больше не нужна
    result_df = result_df.drop(columns=['Original_Col_Order'])
    
    # --- СОХРАНЕНИЕ В EXCEL ---
    
    result_df.to_excel(output_name, index=False)
    
    print(f"Готово! Данные сохранены в файл: {output_name}")
    print(f"Всего получено валидных строк: {len(result_df)}")
    
    return result_df

# --- ЗАПУСК ПРОГРАММЫ ---
if __name__ == "__main__":
    df_clean = convert_excel_to_clean_excel('raw_data.xlsx', 'clean_data.xlsx')
    display(df_clean.head(15))